# 🐍 AI Travel Agent with LangGraph (Python)

## 📋 Scenario Overview

This notebook shows how to build a travel planning agent using **LangGraph** and an Azure AI Foundry (Azure OpenAI) model. The agent can pick a random destination via a custom tool and generate a personalized one-day itinerary.

**Key Features:**
- 🎲 Random destination selection via a custom LangChain tool
- 🧠 Tool-aware reasoning loop powered by LangGraph
- 🔗 Azure AI Foundry model integration (OpenAI-compatible)
- 🧩 Structured agent state (messages) for iterative reasoning

## 🏗️ Technical Implementation

**Core Components:**
- **LangGraph**: Builds a graph-based agent execution flow
- **LangChain**: Provides model + tool abstractions
- **Azure AI Foundry (Azure OpenAI)**: Supplies the chat model
- **Tool**: `get_random_destination()` exposed to the model

### Architecture Flow
```text
User Request → LangGraph Agent (LLM + Tools) → Tool Call (random destination)
                                        ↖︎ reasoning loop ↗︎
                          Final Itinerary Response
```

### Key Elements
- `create_react_agent(...)`: Builds a ReAct-style agent with tool usage
- `Tool`: Wrapper for our Python function
- `AzureChatOpenAI`: Azure-hosted model client

## ⚙️ Prerequisites & Setup

**Dependencies:**
```bash
pip install langgraph langchain langchain-openai python-dotenv azure-identity -U
```

**Environment (.env):**
```env
AZURE_AI_FOUNDRY_ENDPOINT=https://your-endpoint.openai.azure.com
AZURE_AI_FOUNDRY_API_KEY=your_azure_openai_key
AZURE_AI_FOUNDRY_MODEL_ID=gpt-4o-mini        # Your deployment name
AZURE_OPENAI_API_VERSION=2024-07-01-preview  # Adjust if needed
```

**Azure Authentication Options:**
1. Use an API key (simplest) — ensure it's assigned to the deployment.
2. (Optional) Managed Identity / AAD token flows (requires extra setup and current SDK support).

## 🚀 Usage Steps
1. Install dependencies
2. Import libraries & load environment
3. Define the destination tool
4. Initialize the Azure model
5. Build a LangGraph ReAct agent with the tool
6. Invoke the agent with a user request
7. Display the generated travel plan

Let's build your LangGraph travel planner! 🌍✨

### 📦 Imports & Setup

In [10]:
import os
from random import randint
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.prebuilt import create_react_agent

load_dotenv()

True

### 🎲 Tool: Random Destination Selector

In [18]:
def get_random_destination(*args, **kwargs) -> str:
    """Return a random vacation destination from a curated list."""
    destinations = [
        "Barcelona, Spain",
        "Paris, France", 
        "Berlin, Germany",
        "Tokyo, Japan",
        "Sydney, Australia",
        "New York, USA",
        "Cairo, Egypt",
        "Cape Town, South Africa",
        "Rio de Janeiro, Brazil",
        "Bali, Indonesia"
    ]
    return destinations[randint(0, len(destinations) - 1)]

# Update the tool with the fixed function
destination_tool = Tool(
    name="get_random_destination",
    func=get_random_destination,
    description="Returns the name of a random global travel destination for itinerary planning."
)

### 🤖 Initialize Azure Chat Model

In [19]:
azure_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
azure_api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
# Support either AZURE_AI_FOUNDRY_MODEL_ID or legacy AZURE_AI_FOUNDRY_MODEL
azure_deployment = os.getenv("AZURE_AI_FOUNDRY_MODEL_ID") or os.getenv("AZURE_AI_FOUNDRY_MODEL")
api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-07-01-preview")

# Debug prints (mask key)
print(f"Azure Endpoint: {azure_endpoint}")
print(f"Azure Deployment: {azure_deployment}")
print(f"API Version: {api_version}")

if not all([azure_endpoint, azure_api_key, azure_deployment]):
    raise ValueError("Missing one or more required environment variables: AZURE_AI_FOUNDRY_ENDPOINT, AZURE_AI_FOUNDRY_API_KEY, AZURE_AI_FOUNDRY_MODEL_ID")

llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    azure_deployment=azure_deployment,
    api_version=api_version,
    temperature=0.7,
)

Azure Endpoint: https://ibecfoundry.openai.azure.com/
Azure Deployment: gpt-4o
API Version: 2024-02-01


In [20]:
# ✅ Quick sanity check: call the raw model before building the agent
from langchain_core.messages import HumanMessage

try:
    test_resp = llm.invoke([HumanMessage(content="Return just the word TEST")])
    print("Model test OK ->", test_resp.content)
except Exception as e:
    print("Model test failed:", e)
    raise

Model test OK -> TEST


### 🧠 Create LangGraph ReAct Agent

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_INSTRUCTIONS = (
    "You are a helpful travel planning AI. When asked to plan a day trip, "
    "(1) pick a random destination using the get_random_destination tool, then "
    "(2) produce a structured itinerary with morning, afternoon, evening, local cuisine, and tips."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_INSTRUCTIONS),
    MessagesPlaceholder(variable_name="messages")  # the running conversation state that LangGraph manages
])

agent = create_react_agent(
    model=llm,
    tools=[destination_tool],
    prompt=prompt
)

C:\Users\ivbeljan\AppData\Local\Temp\ipykernel_46388\3561065440.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


### 🚀 Run the Agent

In [22]:
user_request = "Plan me a day trip"
# state_modifier already applied; only send the human request.
try:
    result = agent.invoke({"messages": [HumanMessage(content=user_request)]})
except Exception as e:
    print("Agent invocation failed:", e)
    raise

final_messages = result.get("messages", [])
final_ai = next((m for m in reversed(final_messages) if isinstance(m, AIMessage)), None)

if final_ai:
    print("🏖️ Travel Plan:")
    print(final_ai.content)
else:
    print("No AI response found. Messages returned:")
    for m in final_messages:
        print(type(m).__name__, '->', getattr(m, 'content', getattr(m, 'text', '')))

🏖️ Travel Plan:
Here is a structured itinerary for a day trip to Paris, France:

### Morning
- **Start your day at the Eiffel Tower**: Begin the morning with a visit to Paris' iconic landmark. Take an elevator or climb the stairs to enjoy breathtaking views of the city.
- **Stroll along the Seine River**: After your Eiffel Tower visit, take a leisurely walk along the Seine. Stop by the Pont Alexandre III, one of the most beautiful bridges in Paris.

### Afternoon
- **Explore the Louvre Museum**: Spend time wandering through this world-famous museum, home to incredible art pieces such as the Mona Lisa and the Venus de Milo. Allocate at least 2–3 hours.
- **Lunch at a local bistro**: Enjoy a classic French meal, such as a croque monsieur or salade niçoise, paired with a glass of wine.

### Evening
- **Visit Montmartre and the Sacré-Cœur Basilica**: Head to this bohemian neighborhood filled with artistic charm. Admire the panoramic views from the steps of the Sacré-Cœur as the sun sets.
-

### 🔁 (Optional) Streaming / Step-wise Execution

In [ ]:
# Streaming with additional debug info
for update in agent.stream({"messages": [HumanMessage(content="Plan me a food-focused day trip")]}, stream_mode="updates"):
    print("--- Update ---")
    msgs = update.get("messages", [])
    if msgs:
        last = msgs[-1]
        print(type(last).__name__, "=>", getattr(last, 'content', getattr(last, 'text', '')))

### ✅ Next Ideas & Troubleshooting

**If it still fails, check these:**
1. Environment variable names: you must set either `AZURE_AI_FOUNDRY_MODEL_ID` or `AZURE_AI_FOUNDRY_MODEL` to your deployment name (e.g. `gpt-4o-mini`).
2. Endpoint format: should look like `https://<your-resource>.openai.azure.com` (Azure OpenAI) or the Foundry inference endpoint.
3. API key: Ensure the key belongs to the Azure OpenAI resource (not a generic Azure key).
4. Version mismatch: `langgraph` 1.0.x pairs best with recent `langchain` & `langchain-openai`. Run:
   ```python
   import langgraph, langchain_openai, langchain
   print(langgraph.__version__, langchain_openai.__version__, langchain.__version__)
   ```
5. Direct model test cell must return `TEST`. If not, focus on credentials first.
6. Tool visibility: The function name must match exactly the tool name (`get_random_destination`).
7. Network restrictions: Corporate firewalls/proxies may block outbound calls; try a plain curl or run in a different network.

**Escalation Path:**
- If the model test fails: fix env vars / key.
- If model test passes but agent fails: downgrade/upgrade `langgraph` (`pip install langgraph==1.0.1`) to see if regression.
- If streaming hangs: remove `stream_mode="updates"` to isolate.

**Enhancements to try next:**
- Add a weather tool that mocks data.
- Persist chosen destinations to a simple JSON memory file.
- Return structured JSON by instructing the model: "Respond ONLY with JSON schema {...}".
- Integrate RAG by adding a retrieval tool.
